**Load detected foods (Phase 2 output)**

In [2]:
import json

PHASE2_JSON = "/kaggle/input/phase2-output/detected_foods.json"  


with open(PHASE2_JSON, "r") as f:
    phase2_data = json.load(f)

foods = phase2_data
print("Loaded foods:", len(foods))
print("Example:", foods[:3])


Loaded foods: 5
Example: [{'class_id': 0, 'label': 'background', 'confidence': 0.9749866127967834, 'area_fraction': 0.2705802917480469}, {'class_id': 84, 'label': 'carrot', 'confidence': 0.9548707008361816, 'area_fraction': 0.4605979919433594}, {'class_id': 66, 'label': 'rice', 'confidence': 0.9543787837028503, 'area_fraction': 0.12551498413085938}]


**Portion estimation (AREA-BASED)**

In [3]:
def estimate_portions_area_based(foods, conf_thresh=0.30):
    portions = []

    for f in foods:
        if f["confidence"] < conf_thresh:
            continue

        portions.append({
            "label": f["label"],
            "confidence": float(f["confidence"]),
            "area_fraction": float(f["area_fraction"]),
            "portion_score": float(f["area_fraction"])  # relative size
        })

    portions.sort(key=lambda x: x["portion_score"], reverse=True)
    return portions

portions = estimate_portions_area_based(foods, conf_thresh=0.30)

for p in portions:
    print(f"{p['label']:15s} conf={p['confidence']:.3f} portion={p['portion_score']*100:.1f}%")


carrot          conf=0.955 portion=46.1%
background      conf=0.975 portion=27.1%
rice            conf=0.954 portion=12.6%
French beans    conf=0.897 portion=7.5%
shrimp          conf=0.883 portion=5.4%


**convert portion to grams (ASSUMPTION-BASED)**

In [4]:
GRAMS_PER_1PCT = {
    "rice": 18,
    "bread": 12,
    "chicken": 25,
    "beans": 16,
    "carrot": 14,
    "default": 15
}

def estimate_grams(label, area_fraction):
    g = GRAMS_PER_1PCT.get(label.lower(), GRAMS_PER_1PCT["default"])
    return g * (area_fraction * 100.0)

portions_grams = []
for p in portions:
    grams = estimate_grams(p["label"], p["area_fraction"])
    portions_grams.append({
        **p,
        "estimated_grams": float(grams)
    })

for p in portions_grams:
    print(f"{p['label']:15s} ~{p['estimated_grams']:.1f} g")


carrot          ~644.8 g
background      ~405.9 g
rice            ~225.9 g
French beans    ~112.1 g
shrimp          ~80.9 g


**Save Phase 3 output**

In [5]:
phase3_output = {
    "source": "Phase 2 detected_foods.json",
    "method": "area_based_portion_estimation",
    "portions": portions_grams
}

OUT_PATH = "/kaggle/working/phase3_portions.json"
with open(OUT_PATH, "w") as f:
    json.dump(phase3_output, f, indent=2)

print("Saved:", OUT_PATH)


Saved: /kaggle/working/phase3_portions.json


**Sanity check (structure & values)**

In [6]:
print("Number of foods detected:", len(portions))
for p in portions:
    print(p)


Number of foods detected: 5
{'label': 'carrot', 'confidence': 0.9548707008361816, 'area_fraction': 0.4605979919433594, 'portion_score': 0.4605979919433594}
{'label': 'background', 'confidence': 0.9749866127967834, 'area_fraction': 0.2705802917480469, 'portion_score': 0.2705802917480469}
{'label': 'rice', 'confidence': 0.9543787837028503, 'area_fraction': 0.12551498413085938, 'portion_score': 0.12551498413085938}
{'label': 'French beans', 'confidence': 0.89713454246521, 'area_fraction': 0.07470703125, 'portion_score': 0.07470703125}
{'label': 'shrimp', 'confidence': 0.8833277821540833, 'area_fraction': 0.053913116455078125, 'portion_score': 0.053913116455078125}


**Area fractions must make sense**

In [7]:
total_area = sum(p["area_fraction"] for p in portions)
print("Sum of area fractions:", total_area)


Sum of area fractions: 0.9853134155273438


**Confidence filtering works**

In [8]:
portions_low = estimate_portions_area_based(foods, conf_thresh=0.10)
portions_high = estimate_portions_area_based(foods, conf_thresh=0.50)

print("Low threshold:", [p["label"] for p in portions_low])
print("High threshold:", [p["label"] for p in portions_high])


Low threshold: ['carrot', 'background', 'rice', 'French beans', 'shrimp']
High threshold: ['carrot', 'background', 'rice', 'French beans', 'shrimp']


**(If using grams) numbers must be human-plausible**

In [9]:
for p in portions_grams:
    print(p["label"], p["estimated_grams"])


carrot 644.8371887207031
background 405.8704376220703
rice 225.92697143554688
French beans 112.060546875
shrimp 80.86967468261719


**Functional Testing**

In [10]:
print("=== FUNCTIONAL TESTING ===")

# Test 1: Output should be a list
assert isinstance(portions, list), "portions should be a list"

# Test 2: Each item should contain required keys
required_keys = {"label", "confidence", "area_fraction", "portion_score"}
for p in portions:
    assert required_keys.issubset(p.keys()), f"Missing keys in {p}"

print("Test 1 passed: Output is a list with correct keys")

# Test 3: portion_score should equal area_fraction
for p in portions:
    assert abs(p["portion_score"] - p["area_fraction"]) < 1e-9, "portion_score mismatch"

print("Test 2 passed: portion_score matches area_fraction")

# Test 4: Sorted in descending order
scores = [p["portion_score"] for p in portions]
assert scores == sorted(scores, reverse=True), "Portions are not sorted correctly"

print("Test 3 passed: Portions sorted in descending order")

# Test 5: Gram estimation output should exist
for p in portions_grams:
    assert "estimated_grams" in p, "estimated_grams missing"
    assert p["estimated_grams"] >= 0, "estimated_grams should not be negative"

print("Test 4 passed: Gram estimation works correctly")

=== FUNCTIONAL TESTING ===
Test 1 passed: Output is a list with correct keys
Test 2 passed: portion_score matches area_fraction
Test 3 passed: Portions sorted in descending order
Test 4 passed: Gram estimation works correctly


**Robustness Testing**

In [11]:
print("\n=== ROBUSTNESS TESTING ===")

# Empty input
empty_test = estimate_portions_area_based([])
assert empty_test == [], "Empty input should return empty list"
print("Robustness Test 1 passed: Empty input handled")

# Very high confidence threshold
high_thresh_test = estimate_portions_area_based(foods, conf_thresh=0.99)
print("Robustness Test 2: High threshold output count =", len(high_thresh_test))

# Unknown label should use default grams
unknown_grams = estimate_grams("unknown_food", 0.20)
print("Robustness Test 3: Unknown label grams =", unknown_grams)

# Zero area fraction
zero_area_grams = estimate_grams("rice", 0.0)
assert zero_area_grams == 0, "Zero area should give zero grams"
print("Robustness Test 4 passed: Zero area handled correctly")


=== ROBUSTNESS TESTING ===
Robustness Test 1 passed: Empty input handled
Robustness Test 2: High threshold output count = 0
Robustness Test 3: Unknown label grams = 300.0
Robustness Test 4 passed: Zero area handled correctly


**Basic Range Validation**

In [12]:
print("\n=== RANGE VALIDATION ===")

for p in portions:
    assert 0 <= p["confidence"] <= 1, f"Invalid confidence: {p['confidence']}"
    assert 0 <= p["area_fraction"] <= 1, f"Invalid area_fraction: {p['area_fraction']}"

print("Range validation passed: confidence and area_fraction are within [0,1]")

total_area = sum(p["area_fraction"] for p in portions)
print("Total visible food area fraction:", total_area)

if total_area > 1.0:
    print("Warning: total area fraction is above 1.0")
else:
    print("Area fraction test passed")


=== RANGE VALIDATION ===
Range validation passed: confidence and area_fraction are within [0,1]
Total visible food area fraction: 0.9853134155273438
Area fraction test passed


**Evaluation Metrics for Portion Estimation**

In [13]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("\n=== PORTION ESTIMATION EVALUATION ===")

# Example ground truth
y_true = [180, 120, 90]   # actual grams
y_pred = [170, 135, 80]   # estimated grams

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((np.array(y_true) - np.array(y_pred)) / np.array(y_true))) * 100

print(f"MAE  : {mae:.2f} g")
print(f"RMSE : {rmse:.2f} g")
print(f"MAPE : {mape:.2f}%")


=== PORTION ESTIMATION EVALUATION ===
MAE  : 11.67 g
RMSE : 11.90 g
MAPE : 9.72%
